# Faithful vs differentiable selection

`jax-ibart` ships two interchangeable selection backends behind one interface:

- **`bart`** -- faithful BART-G.SE (Bleich 2014 global-SE permutation test) via
  `bartz`. The correctness oracle; matches the reference R iBART.
- **`diff`** -- a differentiable hard-concrete-L0 gate screener (Louizos et al.
  2018). Gradients flow end to end; much faster, lower fidelity on hard problems.

Both must satisfy the PAN criterion (no false negatives in screening).

In [ ]:
import numpy as np
from jax_ibart.selection import BartGSESelector, DiffSelector
import jax

# y depends only on x0, x1 (Friedman); x2..x7 are noise.
rng = np.random.default_rng(0)
X = rng.uniform(0, 1, size=(120, 8))
y = 10 * np.sin(np.pi * X[:, 0] * X[:, 1]) + rng.normal(0, 0.5, 120)

Both backends keep the true drivers and reject noise:

In [ ]:
diff_keep = np.asarray(DiffSelector(steps=1500).select(X, y, key=jax.random.PRNGKey(0)))
bart_keep = np.asarray(BartGSESelector(n_perm=15, n_tree=40, n_draw=300, n_burn=300)
                       .select(X, y, key=jax.random.PRNGKey(0)))
print("diff keeps:", np.where(diff_keep)[0].tolist())
print("bart keeps:", np.where(bart_keep)[0].tolist())
print("truth      : [0, 1]")

### When to use which

| | `bart` (faithful) | `diff` (differentiable) |
|---|---|---|
| Fidelity | high (matches R iBART) | lower on high-dim screens |
| Speed | minutes (CPU) | seconds |
| Differentiable | no (discrete MCMC) | yes, end to end |
| Use for | final/validated results | fast exploration, gradient-based work |

On the real catalysis screen, `diff` is ~13x faster but recovers fewer of the
exact drivers; see [the benchmark](../benchmark.md).